# Study 943 — Reset Frequency ⚖️

**Leveraged ETFs reset their leverage every single evening. Would resetting monthly
instead have been better?**

Every forum thread about TQQQ and UPRO blames the **daily reset** for "volatility decay",
and prescribes the same fix: reset monthly and keep the leverage without the drag. That is
a mechanical claim, so we build the alternative for real — **SPY on margin**, financed at
**^IRX + 50 bps**, levered back to 2x / 3x **once a month** and left to
drift in between — and race it, **excess-of-cash**, against a daily-reset replication and
against **SSO** and **UPRO** themselves.

Windows: 2x on 2007-05-31 → 2026-06-30 (4,799 days, fingerprint
`8f71ea1922ec`); 3x on 2009-06-26 → 2026-06-30 (4,276 days, fingerprint
`4a95749c18bb`). Total-return closes (`auto_adjust=True`); `^IRX` is a *yield*, not a
price. One execution lag: the reset is decided at the month-end close and is in force the
next session.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run only the
offline synthetic control. As-of 2026-06-30.*


## 1. What a reset actually is

A 2x fund is a margin account: for every £1 of yours it holds £2 of the index and owes £1 of cash. If the index falls 10%, your £1 of equity becomes £0.80 while you still hold £1.80 of index — your *effective* leverage has quietly risen from 2.0 to 2.25. Resetting means trading back to exactly 2.0.

**Daily reset** (what SSO and UPRO do): every evening, without fail. **Monthly reset** (the folklore fix): once a month, and whatever the market does in between, you live with it.

## 2. The 2x race — the folklore is *almost* right, and it doesn't matter

In [1]:
R = {'asof': '2026-06-30', 'spread_bps': 50, 'cost_bps': 2, 'maintenance': 25, 'x2_start': '2007-05-31', 'x2_end': '2026-06-30', 'x2_n': 4799, 'x2_fp': '8f71ea1922ec', 'x2_fund_sh': 0.504, 'x2_fund_cagr': 14.33, 'x2_fund_vol': 38.9, 'x2_fund_dd': -84.7, 'x2_fund_term': 12.8, 'x2_day_sh': 0.527, 'x2_day_cagr': 15.45, 'x2_day_vol': 39.6, 'x2_day_dd': -84.2, 'x2_day_term': 15.4, 'x2_mon_sh': 0.549, 'x2_mon_cagr': 17.13, 'x2_mon_vol': 42.8, 'x2_mon_dd': -82.5, 'x2_mon_term': 20.3, 'x2_spy_sh': 0.613, 'x2_spy_cagr': 10.7, 'x2_spy_vol': 19.8, 'x2_spy_dd': -55.2, 'x2_spy_term': 6.9, 'x2_diff_bps': 1.052, 'x2_t': 2.79, 'x2_adv': 0.022, 'x2_adv_ci_lo': -0.01, 'x2_adv_ci_hi': 0.062, 'x2_adv_frac_neg': 8.7, 'x2_mon_ci_lo': 0.168, 'x2_mon_ci_hi': 0.951, 'x2_day_ci_lo': 0.122, 'x2_day_ci_hi': 0.941, 'x2_vs_fund': 0.045, 'x2_t_vs_fund': 3.41, 'x2_fee_leg': 1.21, 'x2_t_fee': 3.82, 'x2_w_mean': 1.998, 'x2_w_min': 1.78, 'x2_w_max': 3.24, 'x2_months': 229, 'x2_slope': -1.25, 'x2_slope_t': -6.83, 'x2_chop': 0.381, 'x2_chop_t': 4.59, 'x2_trend': -0.164, 'x2_trend_t': -8.4, 'x2_pred_slope': 0.115, 'x2_pred_t': 0.72, 'x2_switch': 0.06, 'x2_switch_t': 2.49, 'x2_era_e_adv': 0.047, 'x2_era_e_t': 2.26, 'x2_era_l_adv': 0.004, 'x2_era_l_t': 1.56, 'x2_era_diff_bps': -0.836, 'x2_era_diff_t': -1.15, 'x3_era_diff_bps': -2.369, 'x3_era_diff_t': -0.3, 'x2_sp0_adv': 0.021, 'x2_sp200_adv': 0.025, 'x2_c0_adv': 0.021, 'x2_c10_adv': 0.029, 'x3_start': '2009-06-26', 'x3_end': '2026-06-30', 'x3_n': 4276, 'x3_fp': '4a95749c18bb', 'x3_fund_sh': 0.791, 'x3_fund_cagr': 32.95, 'x3_fund_vol': 51.3, 'x3_fund_dd': -76.8, 'x3_fund_term': 125.5, 'x3_day_sh': 0.808, 'x3_day_cagr': 34.17, 'x3_day_vol': 51.3, 'x3_day_dd': -76.2, 'x3_day_term': 146.6, 'x3_mon_sh': 0.239, 'x3_mon_cagr': 4.05, 'x3_mon_vol': 18.8, 'x3_mon_dd': -48.3, 'x3_mon_term': 2.0, 'x3_spy_sh': 0.911, 'x3_spy_cagr': 15.17, 'x3_spy_term': 11.0, 'x3_diff_bps': -14.678, 'x3_t': -3.62, 'x3_adv': -0.569, 'x3_adv_ci_lo': -1.085, 'x3_adv_ci_hi': 0.017, 'x3_fee_leg': 0.94, 'x3_t_fee': 3.71, 'x3_w_mean': 2.944, 'x3_w_mean_free': 2.983, 'x3_vol_free': 58.5, 'x3_dd_free': -82.2, 'x3_m30_adv': 0.124, 'x3_w_max_free': 8.42, 'x3_w_max_25': 3.84, 'x3_liq': '2011-08-08', 'x3_months': 204, 'x3_slope': -3.158, 'x3_slope_t': -7.22, 'x3_chop': 0.95, 'x3_chop_t': 3.91, 'x3_trend': -0.492, 'x3_trend_t': -8.29, 'x3_pred_slope': 0.659, 'x3_pred_t': 1.61, 'x3_switch': 0.041, 'x3_switch_t': 1.04, 'x3_era_e_adv': -0.581, 'x3_era_e_t': -2.8, 'x3_era_l_adv': -0.563, 'x3_era_l_t': -2.48, 'x3_m0_adv': 0.003, 'x3_m0_term': 227.3, 'x3_m15_adv': -0.285, 'x3_m15_liq': '2020-03-20', 'x3_m30_liq_daily': '2010-05-06', 'st_start': '2004-01-05', 'st_end': '2026-06-30', 'st_n': 5652, 'st_2x_mon': 37.56, 'st_2x_day': 27.91, 'st_3x_mon_free': 80.43, 'st_3x_mon_free_w': 12.8, 'st_3x_mon_25': 0.96, 'st_3x_liq': '2008-09-29', 'st_3x_day': 33.75, 'st_4x_mon_free': 0.0, 'st_4x_liq': '2008-10-24', 'st_4x_day': 17.37, 'syn_chop_gap': 0.357, 'syn_chop_t': 3.74, 'syn_chop_slope': -1.99, 'syn_trend_gap': -0.403, 'syn_trend_t': -2.47, 'syn_trend_slope': -2.65, 'syn_null_gap': 0.033, 'syn_null_t': 0.27, 'syn_null_slope': -2.29}
rows = [('SSO (the real 2x fund)', 'x2_fund'), ('daily-reset replication', 'x2_day'),
        ('MONTHLY-reset replication', 'x2_mon'), ('SPY (1x, reference)', 'x2_spy')]
print(f"{'arm':<26s} {'exSharpe':>9s} {'CAGR':>8s} {'vol':>7s} {'worst DD':>9s} {'x money':>9s}")
for label, k in rows:
    print(f"{label:<26s} {R[k+'_sh']:+9.3f} {R[k+'_cagr']:+7.2f}% {R[k+'_vol']:6.1f}% "
          f"{R[k+'_dd']:8.1f}% {R[k+'_term']:8.1f}x")
print()
print(f"monthly minus daily: {R['x2_diff_bps']:+.2f} bps/day (t = {R['x2_t']:+.2f}) "
      f"-> a REAL extra return")
print(f"but risk-adjusted   : {R['x2_adv']:+.3f} of Sharpe, 95% CI "
      f"[{R['x2_adv_ci_lo']:+.3f}, {R['x2_adv_ci_hi']:+.3f}] -> indistinguishable from zero")

arm                         exSharpe     CAGR     vol  worst DD   x money
SSO (the real 2x fund)        +0.504  +14.33%   38.9%    -84.7%     12.8x
daily-reset replication       +0.527  +15.45%   39.6%    -84.2%     15.4x
MONTHLY-reset replication     +0.549  +17.13%   42.8%    -82.5%     20.3x
SPY (1x, reference)           +0.613  +10.70%   19.8%    -55.2%      6.9x

monthly minus daily: +1.05 bps/day (t = +2.79) -> a REAL extra return
but risk-adjusted   : +0.022 of Sharpe, 95% CI [-0.010, +0.062] -> indistinguishable from zero


The monthly reset turned £1 into **£20.3** where the daily reset made **£15.4** and SSO itself made **£12.8**. That looks like a win — until you notice the volatility went from 39.6% to 42.8% at the same time. Per unit of risk taken, the gain is **+0.022** of Sharpe, and the confidence interval straddles zero.

> 🔬 **For the quants** — the extra return is not free alpha, and it is not extra leverage either: mean exposure is 2.00, the same 2.00 the daily arm runs. It is extra *risk*. Between resets the monthly account's leverage floats — minimum 1.78, **maximum 3.24** (October 2008) — and that float is the whole of the extra volatility.

## 3. Where the difference *comes from* — and why the folklore is upside down

In [2]:
print('monthly-minus-daily gap, by the shape of the month (percentage points/month)')
print(f"  2x: choppy months {R['x2_chop']:+.3f} (t {R['x2_chop_t']:+.2f})  |  "
      f"trending months {R['x2_trend']:+.3f} (t {R['x2_trend_t']:+.2f})")
print(f"  3x: choppy months {R['x3_chop']:+.3f} (t {R['x3_chop_t']:+.2f})  |  "
      f"trending months {R['x3_trend']:+.3f} (t {R['x3_trend_t']:+.2f})")

monthly-minus-daily gap, by the shape of the month (percentage points/month)
  2x: choppy months +0.381 (t +4.59)  |  trending months -0.164 (t -8.40)
  3x: choppy months +0.950 (t +3.91)  |  trending months -0.492 (t -8.29)


In a **choppy** month the daily reset really does rebalance into every reversal, and the monthly reset wins. In a **trending** month the daily reset *compounds the trend* — it adds exposure as you make money — and the monthly reset loses. The daily reset is not the villain of the story; it is the hero of every straight-line month, and the villain only of the zig-zag ones.

> 🔬 **For the quants** — this split is *arithmetic*, not a signal. Section 6 shows the same relationship appears on a pure random walk. And it is not forecastable: use last month's shape to choose this month's reset and the relationship collapses to *t* = +0.72.

## 4. The 3x sleeve — where the story ends abruptly

In [3]:
print(f"3x, {R['x3_start']} -> {R['x3_end']}")
print(f"  UPRO (the real fund)          : x{R['x3_fund_term']:.1f} money, "
      f"excess Sharpe {R['x3_fund_sh']:+.3f}")
print(f"  daily-reset replication       : x{R['x3_day_term']:.1f} money, "
      f"excess Sharpe {R['x3_day_sh']:+.3f}")
print(f"  MONTHLY-reset margin account  : x{R['x3_mon_term']:.1f} money, "
      f"excess Sharpe {R['x3_mon_sh']:+.3f}")
print(f"  ... margin-called on {R['x3_liq']}; the proceeds then sit in cash,")
print("      earning the cash leg and exactly zero excess return, for fifteen years.")

3x, 2009-06-26 -> 2026-06-30
  UPRO (the real fund)          : x125.5 money, excess Sharpe +0.791
  daily-reset replication       : x146.6 money, excess Sharpe +0.808
  MONTHLY-reset margin account  : x2.0 money, excess Sharpe +0.239
  ... margin-called on 2011-08-08; the proceeds then sit in cash,
      earning the cash leg and exactly zero excess return, for fifteen years.


Left alone, the 3x monthly account's leverage drifted to **8.42x** in March 2020. A broker does not leave you alone: at a standard 25% maintenance requirement the position was liquidated on **2011-08-08**, in the post-downgrade selloff, and never came back. (We credit the liquidated account the cash rate afterwards — a margin call should cost you the equity it destroyed, not an invented drag for the next fifteen years.)

And on the longer SPY tape — which reaches the 2008 crash that UPRO, launched in 2009, never saw — a 3x monthly account is called on **2008-09-29**, and a **4x monthly account went to negative equity** on 2008-10-24. A 4x *daily*-reset account, on the same tape, could not: it ended at ×17.4. **That is what the daily reset buys you.**

## 5. So what *is* the monthly reset's edge over the funds?

In [4]:
print(f"monthly reset vs SSO       : {R['x2_vs_fund']:+.3f} Sharpe (t {R['x2_t_vs_fund']:+.2f})")
print(f"  of which reset frequency  : {R['x2_adv']:+.3f}")
print(f"  of which the fund's fee + tracking drag: {R['x2_fee_leg']:+.2f}%/yr "
      f"(t {R['x2_t_fee']:+.2f}) -- nothing to do with resetting")

monthly reset vs SSO       : +0.045 Sharpe (t +3.41)
  of which reset frequency  : +0.022
  of which the fund's fee + tracking drag: +1.21%/yr (t +3.82) -- nothing to do with resetting


About half of it is simply the ~0.9% expense ratio and tracking slippage you avoid by doing it yourself — a real saving, but a *fee* story, not a *reset* story. And it is only yours if you can borrow near the T-bill rate: widen the financing spread to 200 bps and that leg shrinks from +0.056 to +0.010.

## 6. Live check — the machinery is unbiased (offline synthetic)

We plant a world that is deliberately **choppy** (yesterday's move partly reverses), one that is deliberately **trending**, and one that is a pure **random walk** — all with the same volatility. The monthly reset must win in the first, lose in the second, and do nothing at all in the third.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from reset_freq import data, strategy as st
for tag, phi, ss in [('choppy  ', -0.15, 1.0), ('trending', 0.15, 1.0), ('random walk', -0.15, 0.0)]:
    g = [st.synthetic_detect(data.synthetic_daily(phi=phi, signal_strength=ss,
                                                  n_years=12, seed=943+s)[0])
         for s in range(3)]
    print(f"{tag:<12s}: monthly-minus-daily gap {np.mean([d['mean_gap_bps'] for d in g]):+.3f} bps/day"
          f"   (path-shape slope {np.mean([d['chop_slope'] for d in g]):+.2f})")

choppy      : monthly-minus-daily gap +0.326 bps/day   (path-shape slope -1.89)


trending    : monthly-minus-daily gap -0.456 bps/day   (path-shape slope -2.57)


random walk : monthly-minus-daily gap -0.007 bps/day   (path-shape slope -2.20)


Right sign in both planted worlds, **zero on the random walk** — the detector is honest. Note the last column: the path-shape slope is strongly negative *even on the random walk*. That is the proof that the trending-versus-choppy split in section 3 is arithmetic about the path, and never evidence of an edge.

## Verdict

- **Signal — Mixed.** *Real on the return, absent on the Sharpe; positive at 2x, negative at 3x.* Reset frequency genuinely moves returns (2x: +1.05 bps/day, HAC *t* = +2.79) and its direction is real and well identified — but the *claim* that monthly is **better** fails: +0.022 of Sharpe with a CI straddling zero, unforecastable, and -0.569 at 3x. (The second half of the sample looks weaker, +0.004 against +0.047, but the era difference itself is *t* = -1.15 — not a decay we can claim.)
- **Tradability — Mirage.** The monthly reset's real product is uncontrolled leverage — 8.42x at the peak, a margin call in 2011, negative equity at 4x in 2008 — all of it at the *same* average leverage the daily arm ran. The daily reset's supposed vice is the feature that keeps you solvent, and it costs about two hundredths of a Sharpe.